In [1]:
import pandas as pd

df = pd.read_csv("../../data/processed/room_price_options.csv", sep=",")
df.head()

,room_type_id,price_option_name,price_option_price,price_amenities_for_price
0,1,Thông tin sức chứa phòng: 2 người lớn.,289.522\r\n₫,['Miễn phí hủy trước 17 tháng 11 2025\n\ntoolt...
1,2,Thông tin sức chứa phòng: 2 người lớn.,361.111\r\n₫,['Miễn phí hủy trước 17 tháng 11 2025\n\ntoolt...
2,3,Thông tin sức chứa phòng: 4 người lớn và 2 trẻ...,601.852\r\n₫,['Miễn phí hủy trước 17 tháng 11 2025\n\ntoolt...
3,4,Thông tin sức chứa phòng: 2 người lớn.,425.926\r\n₫,['Miễn phí hủy trước 17 tháng 11 2025\n\ntoolt...
4,5,Thông tin sức chứa phòng: 2 người lớn.,1.450.000\r\n₫,"['Có bữa sáng (204.120 ₫/người)', 'Chính sách ..."


In [2]:
import re

# extract column option_name to 2 columns: adults_number, children_number
def extract_people(option):
    option = str(option)
    
    # get number of adults
    adults_match = re.search(r'(\d+)\s*người lớn', option)
    adults = int(adults_match.group(1)) if adults_match else 0
    
    # get number of children
    children_match = re.search(r'(\d+)\s*trẻ em', option)
    children = int(children_match.group(1)) if children_match else 0
    
    return pd.Series([adults, children])

df[["adults_number", "children_number"]] = df["price_option_name"].apply(extract_people)

# df.iloc[:, :8]
df[["price_option_name", "adults_number", "children_number"]]

,price_option_name,adults_number,children_number
0,Thông tin sức chứa phòng: 2 người lớn.,2,0
1,Thông tin sức chứa phòng: 2 người lớn.,2,0
2,Thông tin sức chứa phòng: 4 người lớn và 2 trẻ...,4,2
3,Thông tin sức chứa phòng: 2 người lớn.,2,0
4,Thông tin sức chứa phòng: 2 người lớn.,2,0
...,...,...,...
16504,Thông tin sức chứa phòng: 2 người lớn.,2,0
16505,Thông tin sức chứa phòng: 2 người lớn.,2,0
16506,Thông tin sức chứa phòng: 4 người lớn.,4,0
16507,Thông tin sức chứa phòng: 2 người lớn.,2,0


In [3]:
# Remove price_option_name column and clean price_option_price
import re

# Remove the price_option_name column
df = df.drop(columns=['price_option_name'])

# Clean price_option_price to extract only float values
def clean_price(price_str):
    if pd.isna(price_str):
        return None
    # Convert to string and remove newlines, carriage returns, and the 'đ' character
    price_str = str(price_str)
    price_str = price_str.replace('\r', '').replace('\n', '').replace('đ', '').strip()
    
    # Remove all non-numeric characters except dots and commas
    price_str = re.sub(r'[^\d.,]', '', price_str)
    
    # If there are dots, check if they're thousand separators (e.g., 200.000)
    if '.' in price_str:
        # Split by dot to check format
        parts = price_str.split('.')
        # If last part has 3 digits, dots are likely thousand separators
        if len(parts) > 1 and len(parts[-1]) == 3:
            # Remove all dots (thousand separators)
            price_str = price_str.replace('.', '')
        # Otherwise, treat as decimal separator
        elif ',' in price_str:
            # If comma exists, it might be decimal separator
            price_str = price_str.replace('.', '').replace(',', '.')
    
    # Convert to float
    try:
        return float(price_str)
    except (ValueError, AttributeError):
        return None

df['price_option_price'] = df['price_option_price'].apply(clean_price)
df.head()

,room_type_id,price_option_price,price_amenities_for_price,adults_number,children_number
0,1,289522.0,['Miễn phí hủy trước 17 tháng 11 2025\n\ntoolt...,2,0
1,2,361111.0,['Miễn phí hủy trước 17 tháng 11 2025\n\ntoolt...,2,0
2,3,601852.0,['Miễn phí hủy trước 17 tháng 11 2025\n\ntoolt...,4,2
3,4,425926.0,['Miễn phí hủy trước 17 tháng 11 2025\n\ntoolt...,2,0
4,5,1450000.0,"['Có bữa sáng (204.120 ₫/người)', 'Chính sách ...",2,0


In [4]:
# Convert price_amenities_for_price into one-hot encoded columns
import ast      # Evaluate strings look like Python lists into actual Python objects
import re

# Function to parse amenities from the column
def parse_amenities(amenities_str):
    if pd.isna(amenities_str):
        return []
    try:
        # Try convert string to Python object (list)
        amenities = ast.literal_eval(str(amenities_str))
        if isinstance(amenities, list):
            # Clean each amenity string (remove newlines, extra spaces)
            cleaned_amenities = [str(a).replace('\n', ' ').replace('\r', '').strip() for a in amenities]
            return cleaned_amenities
        else:
            return []
    except (ValueError, SyntaxError):
        # If parsing fails, try to extract as comma-separated or handle as single string
        return [str(amenities_str).strip()]

# Function to create a clean column name from amenity string
def clean_column_name(amenity_str):
    # Replace spaces and special characters with underscores
    # Keep only alphanumeric, underscores, and Vietnamese characters
    cleaned = re.sub(r'[^\w\s%]', '', str(amenity_str))
    cleaned = re.sub(r'\s+', '_', cleaned.strip())
    cleaned = cleaned.lower()
    # Limit length to avoid extremely long column names
    return cleaned

# Parse amenities for all rows
df['amenities_list'] = df['price_amenities_for_price'].apply(parse_amenities)

# Get all unique amenities across all rows
all_amenities = set()
for amenities_list in df['amenities_list']:
    all_amenities.update(amenities_list)

# Sort amenities for consistent column ordering
all_amenities = sorted(list(all_amenities))

print(all_amenities)

# Create one-hot encoded columns with clean names
# Store mapping of original amenity to column name
amenity_to_col = {}
for amenity in all_amenities:
    col_name = clean_column_name(amenity)
    # Handle duplicate column names by appending index
    original_col_name = col_name
    counter = 1
    while col_name in amenity_to_col.values():
        col_name = f"{original_col_name}_{counter}"
        counter += 1
    amenity_to_col[amenity] = col_name
    df[col_name] = df['amenities_list'].apply(lambda x: 1 if amenity in x else 0)

# Drop the temporary amenities_list column and original price_amenities_for_price column
df = df.drop(columns=['amenities_list', 'price_amenities_for_price'])

df.head()


['1 người lớn & 1 trẻ em (0-1 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-10 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-11 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-12 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-16 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-17 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-2 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-3 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-4 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-5 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-6 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-7 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-8 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (0-9 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 1 trẻ em (1-6 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 2 trẻ em (0-10 tuổi) Vượt quá sức chứa phòng', '1 người lớn & 2 

C:\Users\Admin\AppData\Local\Temp\ipykernel_18224\1524596788.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = df['amenities_list'].apply(lambda x: 1 if amenity in x else 0)
C:\Users\Admin\AppData\Local\Temp\ipykernel_18224\1524596788.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = df['amenities_list'].apply(lambda x: 1 if amenity in x else 0)
C:\Users\Admin\AppData\Local\Temp\ipykernel_18224\1524596788.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of

,room_type_id,price_option_price,adults_number,children_number,1_người_lớn_1_trẻ_em_01_tuổi_vượt_quá_sức_chứa_phòng,1_người_lớn_1_trẻ_em_010_tuổi_vượt_quá_sức_chứa_phòng,1_người_lớn_1_trẻ_em_011_tuổi_vượt_quá_sức_chứa_phòng,1_người_lớn_1_trẻ_em_012_tuổi_vượt_quá_sức_chứa_phòng,1_người_lớn_1_trẻ_em_016_tuổi_vượt_quá_sức_chứa_phòng,1_người_lớn_1_trẻ_em_017_tuổi_vượt_quá_sức_chứa_phòng,...,đã_áp_dụng_chiết_khấu_chiến_dịch_thúc_đẩy_94736,đón_khách_tại_sân_bay,đưa_ra_sân_bay,đưa_đón_sân_bay_hai_chiều,đưa_đón_sân_bay_một_chiều,đưa_đón_sân_bay_hai_chiều_1,đặt_không_cần_thẻ_tín_dụng,đặt_và_trả_tiền_ngay,đồ_dùng_cho_khách_nữ,đồ_uống_không_giới_hạn_có_cồn_cho_2_người
0,1,289522.0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,2,361111.0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,3,601852.0,4,2,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,4,425926.0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,5,1450000.0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [5]:
import json

amenities_list_json = list(all_amenities)
with open("ooriginal_all_amenities.json", "w", encoding="utf-8") as f:
    json.dump(amenities_list_json, f, ensure_ascii=False, indent=4)

In [6]:
# Remove columns containing at least 1 element in the list of string
def drop_columns_contaning(df, keywords):
    pattern = "|".join(map(str, keywords))
    cols_to_remove = df.columns[df.columns.str.contains(pattern, case=False, regex=True)]
    return df.drop(columns=cols_to_remove)

# Merge all columns containing the input string
def merge_and_replace_columns(df, keyword):
    cols = df.columns[df.columns.str.contains(keyword, case=False, regex=True)]
    if len(cols) == 0:
        return df
    # If any column is NaN, consider it as 0
    group = df[cols].fillna(0)
    # Create a new column
    df[keyword] = group.any(axis=1).astype(int)
    # Remove old columns
    df = df.drop(columns=cols)
    return df


import re
import pandas as pd

# Function for extracting numeric features
def extract_numeric_feature(df: pd.DataFrame, keyword: str, pattern: str):
    kw = keyword.lower()

    # Find all columns containing keyword
    selected_cols = [c for c in df.columns if kw in c.lower()]
    if not selected_cols:
        return df

    # Get numeric value from column's name
    def get_value(colname):
        m = re.search(pattern, colname.lower())
        # return int(m.group(1)) if m else 0
        return int(m.group(1).replace(".", "")) if m else 0

    # Map: {column's name: numeric value}
    col_value_map = {c: get_value(c) for c in selected_cols}

    # Create new column
    new_col_name = (
        keyword.lower()
               .replace(" ", "_")
               .replace("đ", "d")
    )

    # Determine row having column's value = 1 and mapping
    df[new_col_name] = df[selected_cols].idxmax(axis=1).map(col_value_map)

    # Remove old columns
    df.drop(columns=selected_cols, inplace=True)

    return df

# Combine columns
def combine_columns(df, source_cols, new_col, drop_source=True):
    # Only take columns that exist
    source_cols = [c for c in source_cols if c in df.columns]
    if not source_cols:
        return df

    # OR logic: if there exist at least one column not 0 -> new value = 1
    df[new_col] = (df[source_cols] != 0).any(axis=1).astype(int)
    if drop_source:
        df = df.drop(columns=source_cols)

    return df

In [7]:
percentage_pattern = r"(\d+)\s*%"
chiec_pattern = r"(\d+)\s*chiếc"
tieng_pattern =  r"(\d+)\s*tiếng"
phut_pattern = r"(\d+)\s*phút"
# price_pattern = r"(\d[\d\.]*)\s*đ"
price_pattern = r"(\d[\d\.]*)[\s\u00A0]*[đ₫]"

am_pm_pattern = r"(\d{1,2}\s*[AP]M)"
vnd_pattern = r"(\d+)\s*VND"

In [8]:
# REMOVE UNECESSARY COLUMNS

# Remove columns containing "người_lớn" or "trẻ_em"
df = drop_columns_contaning(df, ["người_lớn", 'trẻ_em'])
# Remove columns containing "xét_nghiệm_RT_PCR"
df = drop_columns_contaning(df, ["xét_nghiệm_RTPCR"])
# Remove columns "báo_cáo_giấy_chứng_nhận_y_tế"
df = df.drop(columns=["báo_cáogiấy_chứng_nhận_y_tế"], errors="ignore")
# Remove column "bao_gồm_2_bữa_ăn"
df = df.drop(columns=["bao_gồm_2_bữa_ăn"], errors="ignore")
# Remove columns "bữa_sáng_chay_cho_2_người", "bữa_sáng_cho_2_người", "bữa_sáng_món_chay", "bữa_sáng_địa_phương"
df = df.drop(columns=["bữa_sáng_chay_cho_2_người", "bữa_sáng_cho_2_người", "bữa_sáng_món_chay", "bữa_sáng_địa_phương"], errors="ignore")
# Remove column "cho_phép_giao_hàng_từ_gia_đình_và_họ_hàng"
df = df.drop(columns=["cho_phép_giao_hàng_từ_gia_đình_và_họ_hàng"], errors="ignore")
# Merge columns containing "rượu"
df = merge_and_replace_columns(df, "rượu")
# Remove column "Chính sách hủy"
df = df.drop(columns=["chính_sách_hủy"], errors="ignore")
# Remove column "Chương trình bộ sưu tập khách sạn đối tác"
df = df.drop(columns=["chương_trình_bộ_sưu_tập_khách_sạn_đối_tác"], errors="ignore")
# Remove column "Chấp nhận giấy tờ tùy thân địa phương"
df = df.drop(columns=["chấp_nhận_giấy_tờ_tùy_thân_địa_phương"], errors="ignore")
# Remove column "Chỉ Cho Nhân Viên Then Chốt"
df = df.drop(columns=["chỉ_cho_nhân_viên_then_chốt"], errors="ignore")
# Merge columns "Chỉ Ở ... Tiếng"
df = extract_numeric_feature(df, "chỉ_ở", tieng_pattern)
# Remove column "Các hoạt động dưới nước", "Các môn thể thao nước chọn lọc"
df = df.drop(columns=["các_hoạt_động_dưới_nước", "các_môn_thể_thao_nước_chọn_lọc"], errors="ignore")
# Merge columns "Có bữa sáng (... đ/người)"
df = extract_numeric_feature(df, "có_bữa_sáng", price_pattern)
# Remove column "Có ăn sáng"
df = df.drop(columns=["có_ăn_sáng"], errors="ignore")
# Merge columns "Giảm giá Giặt là ...%"
df = extract_numeric_feature(df, "giảm_giá_giặt_là", percentage_pattern)
# Drop column "Giảm gia spa"
df = df.drop(columns=["giảm_giá_spa"])
# Merge columns "Giảm giá Spa ...%"
df = extract_numeric_feature(df, "giảm_giá_spa", percentage_pattern)
# Remove column "Giảm giá Thức ăn & Đồ uống (%)"
df = df.drop(columns=["giảm_giá_thức_ăn_đồ_uống"], errors="ignore")
# Merge columns "Giảm giá Thức ăn & Đồ uống ...%"
df = extract_numeric_feature(df, "giảm_giá_thức_ăn_đồ_uống", percentage_pattern)
# Remove column "Giảm giá dúng bữa", "Giảm giá đồ ăn", "Gimar giá đồ ăn uống", "Giảm giá Đồ uống 10%"
df = df.drop(columns=["giảm_giá_dùng_bữa", "giảm_giá_đồ_ăn", "giảm_giá_đồ_ăn_uống", "giảm_giá_đồ_uống_10"], errors="ignore")
# Remove column "Giặt là miễn phí"
df = df.drop(columns=["giặt_là_miễn_phí"], errors="ignore")
# Merge columns "Giặt là miễn phí (Mỗi ngày) ... chiếc"
df = extract_numeric_feature(df, "giặt_là_miễn_phí", chiec_pattern)
# Remove column "Giỏ trái cây"
df = df.drop(columns=["giỏ_trái_cây"], errors="ignore")
# Remove columns containing "Không cần thanh toán đến"
df = drop_columns_contaning(df, ["không_cần_thanh_toán_đến"])
# Remove column "Không gluten"
df = df.drop(columns=["không_gluten"], errors="ignore")
# Merge columns containing "Không hoàn tiền"
df = merge_and_replace_columns(df, "không_hoàn_tiền")
# Merge columns containing "Không phải thanh toán ngay"
df = merge_and_replace_columns(df, "không_phải_thanh_toán_ngay")
# Merge columns containing "Liệu pháp spa ... phút"
df = extract_numeric_feature(df, "liệu_pháp_spa", phut_pattern)
# Merge columns containing "Miễn phí hủy bỏ"
df = merge_and_replace_columns(df, "miễn_phí_hủy")
# Merge columns containing "Mát-xa chân ... phút"
df = extract_numeric_feature(df, "mátxa_chân", phut_pattern)
# Remove column "Nhân viên y tế gọi điện"
df = df.drop(columns=["nhân_viên_y_tế_gọi_điện"])
# Merge columns containing "Nhận Phòng Muộn sau ... AM/PM"
df = extract_numeric_feature(df, "nhận_phòng_muộn_sau", am_pm_pattern)
# Remove columns "Nhận phòng nhanh", "Nhận phòng sớm", "Nhận phòng trễ"
df = df.drop(columns=["nhận_phòng_nhanh", "nhận_phòng_sớm", "nhận_phòng_trễ"], errors="ignore")
# Merge columns containing "Nhận phòng sớm từ ... AM/PM"
df = extract_numeric_feature(df, "nhận_phòng_sớm_từ", am_pm_pattern)
# Remove column "Nâng cấp lên loại phòng cao hơn", "Phòng được giao sẽ là phòng cơ bản nhất hoặc tốt hơn"
df = df.drop(columns=["nâng_cấp_lên_loại_phòng_cao_hơn", "phòng_được_giao_sẽ_là_phòng_cơ_bản_nhất_hoặc_tốt_hơn"], errors="ignore")
# Merge columns containing "Phục vụ bữa sáng (... đ / người)"
df = extract_numeric_feature(df, "phục_vụ_bữa_sáng", price_pattern)
# Remove column "Quyền lui tới khu giải trí có điều kiện"
df = df.drop(columns=["quyền_lui_tới_khu_giải_trí_có_điều_kiện"], errors="ignore")
# Remove column "Quả tạ"
df = df.drop(columns=["quả_tạ"], errors="ignore")
# Remove column "Spa discount (%)", "Số tiền dùng tại spa", "Số tiền để đậu xe"
df = df.drop(columns=["spa_discount_%", "số_tiền_dùng_tại_spa", "số_tiền_để_đậu_xe"], errors="ignore")
# Remove column "Thanh toán tại nơi ở"
df = df.drop(columns=["thanh_toán_tại_nơi_ở"], errors="ignore")
# Remove column "Thức uống"
df = df.drop(columns=["thức_uống"], errors="ignore")
# Remove column "Trả phòng trễ trước 6 PM"
df = df.drop(columns=["trả_phòng_trễ_trước_6_pm"], errors="ignore")
# Remove column "Trả phòng muộn"
df = df.drop(columns=["trả_phòng_muộn"], errors="ignore")
# Merge columns containing "Trả phòng muộn đến ...AM/PM"
df = extract_numeric_feature(df, "trả_phòng_muộn_đến", am_pm_pattern)
# Remove column "Trả phòng trễ", "Trả tiền cho khách sạn", "Trả tiền ở khách sạn"
df = df.drop(columns=["trả_phòng_trễ", "trả_tiền_cho_khách_sạn", "trả_tiền_ở_khách_sạn"], errors="ignore")
# Merge columns containing "TÍn dụng Nhà hàng ... VND"
df = extract_numeric_feature(df, "tín_dụng_nhà_hàng", vnd_pattern)
# Merge columns containing "TÍn dụng Spa ... VND"
df = extract_numeric_feature(df, "tín_dụng_spa", vnd_pattern)
# Merge columns containing "TÍn dụng Thức ăn và Đồ uống  ... VND"
df = extract_numeric_feature(df, "tín_dụng_thức_ăn_và_đồ_uống", vnd_pattern)
# Merge columns containing "TÍn dụng Đồ uống ... VND"
df = extract_numeric_feature(df, "tín_dụng_đồ_uống", vnd_pattern)
# Merge columns containing "Tín dụng Đỗ xe ... VND"
df = extract_numeric_feature(df, "tín_dụng_đỗ_xe", vnd_pattern)
# Remove columns "Vào CLB thể thao Banyan Tree miễn phí", "Vào Club lounge", "Vào hồ bơi Vô cực miễn phí"
df = df.drop(columns=["vào_clb_thể_thao_banyan_tree_miễn_phí", "vào_club_lounge", "vào_hồ_bơi_Vô_cực_miễn_phí"], errors="ignore")
# Remove column "Vé miễn phí"
df = df.drop(columns=["vé_miễn_phí"], errors="ignore")
# Remove column "Đã gồm bữa sáng ngon", "Đã gồm bữa sáng rất ngon", "Đã gồm bữa sáng tuyệt hảo"
df = df.drop(columns=["đã_gồm_bữa_sáng_ngon", "đã_gồm_bữa_sáng_rất_ngon", "đã_gồm_bữa_sáng_tuyệt_hảo"], errors="ignore")
# Merge columns containing "đã_gồm_bữa_sáng"
df = extract_numeric_feature(df, "đã_gồm_bữa_sáng", percentage_pattern)
# Merge columns containing "Đã nhập mã AGODA_SPONSORED - giảm ... đ!"
df = extract_numeric_feature(df, "đã_nhập_mã_agoda_sponsored", price_pattern)
# Merge columns containing "Đã áp dụng chiết khấu Chiến dịch Thúc đẩy: ... đ"
df = extract_numeric_feature(df, "chiết_khấu_chiến_dịch_thúc_đẩy", price_pattern)
# Remove columns "Đưa đón sân bay (hai chiều)", "Đưa đón sân bay (một chiều)", "Đưa đón sân bay [hai chiều]"
df = df.drop(columns=["đưa_đón_sân_bay_hai_chiều", "đưa_đón_sân_bay_một_chiều"], errors="ignore")

C:\Users\Admin\AppData\Local\Temp\ipykernel_18224\3839557473.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[keyword] = group.any(axis=1).astype(int)
C:\Users\Admin\AppData\Local\Temp\ipykernel_18224\3839557473.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col_name] = df[selected_cols].idxmax(axis=1).map(col_value_map)
C:\Users\Admin\AppData\Local\Temp\ipykernel_18224\3839557473.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

In [9]:
df = df.drop(columns=["giảm_giá_đồ_uống_10%"], errors="ignore")
df = df.drop(columns=["mátxa_30_phút"], errors="ignore")
df = df.drop(columns=["tra_phong_trễ_trước_6_pm"], errors="ignore")
df = df.drop(columns=["vào_hồ_bơi_vô_cực_miễn_phí"], errors="ignore")
df = df.drop(columns=["đưa_đón_sân_bay_hai_chiều_1"], errors="ignore")
df = df.drop(columns=["chỉ_ở"], errors="ignore")
df = df.drop(columns=["spa_discount_%"], errors="ignore")
df = extract_numeric_feature(df, "chính_sách_hủy_tooltip", percentage_pattern)

df = combine_columns(df, ["bữa_sáng_miễn_phí", "kèm_bữa_sáng"], "đã_kèm_bữa_sáng")
df = combine_columns(df, ["bao_gồm_bữa_trưa", "bữa_trưa_miễn_phí"], "đã_kèm_bữa_trưa")
df = combine_columns(df, ["bao_gồm_bữa_tối", "bữa_tối_miễn_phí"], "đã_kèm_bữa_tối")

C:\Users\Admin\AppData\Local\Temp\ipykernel_18224\3839557473.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col_name] = df[selected_cols].idxmax(axis=1).map(col_value_map)


In [10]:
df.loc[df["đồ_uống_không_giới_hạn_có_cồn_cho_2_người"] != 0, ["đồ_uống_không_giới_hạn_có_cồn_cho_2_người"]]

,đồ_uống_không_giới_hạn_có_cồn_cho_2_người
1225,1
1230,1
1235,1
11203,1
11214,1
11227,1


In [11]:
pd.set_option('display.max_columns', None)
df.head()
# pd.reset_option('display.max_columns')

,room_type_id,price_option_price,adults_number,children_number,bãi_đậu_xe,chiết_khấu_dịch_vụ_giặt_là,chiết_khấu_dịch_vụ_tại_phòng,cho_phép_order_đồ_ăn_bên_ngoài,các_kênh_truyền_hình_cáp,dịch_vụ_dọn_phòng_có_hạn_chế,dịch_vụ_giặt_ủi_có_giới_hạn,dịch_vụ_tư_vấn_khám_chữa_bệnh_từ_xa,dịch_vụ_xe_đưa_đón_miễn_phí,giao_hàng_từ_cửa_hàng_tiện_lợi_gần_đó,không_hút_thuốc,khẩu_trang_bảo_vệ,lớp_học_yoga_mỗi_ngày,massage_chân,medical_services_discount,miễn_phí_internet_không_dây,miễn_phí_thuê_yukata,miễn_phí_đồ_ăn_nhẹ,máy_chạy_bộ,máy_cà_phê_espresso_có_viên_nén,nhiều_cách_nữa_để_thanh_toán_gồm_cả_paypal,phà_đưa_đón_2_chiều,phòng_họp,phòng_tập,phù_hợp_cho_cặp_đôi,quà_tặng_miễn_phí,quầy_bar_mini_miễn_phí,suối_nước_nóng_tắm_riêng,tv_thông_minh_có_ứng_dụng,thiết_bị_wifi_bỏ_túi,thuê_xe_máy,thuê_xe_ô_tô,thuê_xe_đạp,thảm_yoga,tour_tham_quan,trà_chiều,tín_dụng_khách_sạn,voucher_mua_hàng_miễn_thuế,voucher_resort,voucher_ăn_uống,voucher_đồ_uống,vào_hồ_bơi_miễn_phí,vào_phòng_tập_ngoài_trời,vào_phòng_tập_trong_nhà,vé_công_viên_giải_trí,vé_công_viên_nước,wifi_miễn_phí,wifi_cao_cấp_miễn_phí,xe_đạp_thể_dục,xông_hơi_miễn_phí,ăn_chay,ăn_thuần_chay,đón_khách_tại_sân_bay,đưa_ra_sân_bay,đặt_không_cần_thẻ_tín_dụng,đặt_và_trả_tiền_ngay,đồ_dùng_cho_khách_nữ,đồ_uống_không_giới_hạn_có_cồn_cho_2_người,rượu,có_bữa_sáng,giảm_giá_giặt_là,giảm_giá_spa,giảm_giá_thức_ăn_dồ_uống,giặt_là_miễn_phí,không_hoàn_tiền,không_phải_thanh_toán_ngay,liệu_pháp_spa,miễn_phí_hủy,mátxa_chân,nhận_phòng_muộn_sau,nhận_phòng_sớm_từ,phục_vụ_bữa_sáng,trả_phòng_muộn_dến,tín_dụng_nhà_hàng,tín_dụng_spa,tín_dụng_thức_ăn_và_dồ_uống,tín_dụng_dồ_uống,tín_dụng_dỗ_xe,dã_gồm_bữa_sáng,dã_nhập_mã_agoda_sponsored,chiết_khấu_chiến_dịch_thúc_dẩy,chính_sách_hủy_tooltip,đã_kèm_bữa_sáng,đã_kèm_bữa_trưa,đã_kèm_bữa_tối
0,1,289522.0,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,10,10,0,10,0,0,0,1,0,0,0,0,0,0,0,0,0,0,65,0,0,40,0,0,0
1,2,361111.0,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,10,10,0,10,0,0,0,1,0,0,0,0,0,0,0,0,0,0,65,0,0,40,0,0,0
2,3,601852.0,4,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,10,10,0,10,0,0,0,1,0,0,0,0,0,0,0,0,0,0,65,0,0,40,0,0,0
3,4,425926.0,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,10,10,0,10,0,0,0,1,0,0,0,0,0,0,0,0,0,0,65,0,0,40,0,0,0
4,5,1450000.0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,10,10,0,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,65,0,0,100,0,0,0


In [12]:
for col in df.columns:
    print(col)

room_type_id
price_option_price
adults_number
children_number
bãi_đậu_xe
chiết_khấu_dịch_vụ_giặt_là
chiết_khấu_dịch_vụ_tại_phòng
cho_phép_order_đồ_ăn_bên_ngoài
các_kênh_truyền_hình_cáp
dịch_vụ_dọn_phòng_có_hạn_chế
dịch_vụ_giặt_ủi_có_giới_hạn
dịch_vụ_tư_vấn_khám_chữa_bệnh_từ_xa
dịch_vụ_xe_đưa_đón_miễn_phí
giao_hàng_từ_cửa_hàng_tiện_lợi_gần_đó
không_hút_thuốc
khẩu_trang_bảo_vệ
lớp_học_yoga_mỗi_ngày
massage_chân
medical_services_discount
miễn_phí_internet_không_dây
miễn_phí_thuê_yukata
miễn_phí_đồ_ăn_nhẹ
máy_chạy_bộ
máy_cà_phê_espresso_có_viên_nén
nhiều_cách_nữa_để_thanh_toán_gồm_cả_paypal
phà_đưa_đón_2_chiều
phòng_họp
phòng_tập
phù_hợp_cho_cặp_đôi
quà_tặng_miễn_phí
quầy_bar_mini_miễn_phí
suối_nước_nóng_tắm_riêng
tv_thông_minh_có_ứng_dụng
thiết_bị_wifi_bỏ_túi
thuê_xe_máy
thuê_xe_ô_tô
thuê_xe_đạp
thảm_yoga
tour_tham_quan
trà_chiều
tín_dụng_khách_sạn
voucher_mua_hàng_miễn_thuế
voucher_resort
voucher_ăn_uống
voucher_đồ_uống
vào_hồ_bơi_miễn_phí
vào_phòng_tập_ngoài_trời
vào_phòng_tập_trong_nhà

In [13]:
import os

os.makedirs("processed", exist_ok=True)
df.to_csv("./processed/room_price_options.csv", sep=";", encoding="utf-8-sig", index=False)